# Workshop 04 — Python, NumPy & Linear Regression

**One problem:** Learn a character's horizontal position from animation frames.

Open the matching [interactive lesson](https://siadu.github.io/MLMPworkshop/workshop04/) for visual explanations. This notebook is where you **write and run the code**. Start at Section 00 and run cells in order.

| Section | What you build |
|---|---|
| 00 · Motion data | Read the samples and inspect the plot |
| 01 · Python foundations | Mean and count functions |
| 02 · NumPy & shapes | Arrays, feature shapes, scaling, matrix operations |
| 03 · Prediction & loss | A linear model and its MSE |
| 04 · Gradient descent | Gradients, parameter update, training loop |
| 05 · Multiple features | Predict from `x` and `x²` |
| 06 · Train & compare | Compare models on accelerating motion |

**Naming used everywhere:** `x` = time in seconds (one input feature), `y` = measured horizontal position (target), `y_pred` = prediction, `w` = weight, `b` = bias, `N` = number of samples, `error = y_pred - y` = residual, `loss` = mean squared error (MSE). Capital `X` = input matrix with several features.

## Setup

Download this `.ipynb` file and place it in `MLMPworkshop/workshop04_Python_NumPy_LinearRegression/`. The project should be in `~/Desktop/MLMPworkshop`.

```bash
cd ~/Desktop/MLMPworkshop/workshop04_Python_NumPy_LinearRegression/
ls
# Workshop04_Python_NumPy_LinearRegression.ipynb
uv init
uv venv --python 3.11
source .venv/bin/activate  # Linux / macOS
uv add numpy matplotlib ipykernel notebook
```

Go to `.gitignore` and add `.ipynb_checkpoints` as the last line.


## 00 · Motion Data

A character moves from left to right. We record its horizontal position at 11 frames.

- `frames`: frame numbers from 0 to 30, measured every 3 frames.
- `positions`: horizontal positions measured in scene units.
- `fps = 30`: the sequence uses 30 frames per second.
- Each pair of time and position is one sample.

We use **time in seconds** as our input: `x = frame / fps`. At 30 fps, frame 15 is 0.5 seconds. Converting frames to seconds is not required by linear regression; it gives a meaningful time unit and a smaller numerical input scale for gradient descent.

**Input:** one feature (time). **Target:** one horizontal position per sample. The interactive lesson shows a playable video and the data table.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

frames = [0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30]
positions = [1.52, 2.02, 2.52, 3.01, 3.51, 4.01, 4.51, 5.01, 5.5, 6, 6.5]
fps = 30
assert len(frames) == len(positions)

def plot_data(x, y, title="Observed motion"):
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.scatter(x, y, color="#1f78b4", s=45, label="Measured position")
    ax.set(xlabel="Time x (seconds)", ylabel="Horizontal position y", title=title)
    ax.grid(alpha=0.25)
    ax.legend()
    plt.show()

plot_data(np.array(frames) / fps, np.array(positions))
print("Samples:", len(frames))
print("First sample:", frames[0], "→", positions[0])

**Next:** Return to the interactive lesson, **Section 01 — Python Foundations**.


## 01 · Python Foundations

The interactive lesson shows variable naming rules and examples of `list`, indexing, `len()`, `print()`, `for`, `if` / `elif` / `else`, `def` and `return`.

In the next tasks, use a loop to calculate an answer and a function to reuse it. Your functions should **return** the value, not only print it.


In [ ]:
values = [3, 7, 2]
print("First value:", values[0])        # Index starts at 0
print("Number of values:", len(values))
total = 0
for value in values:
    total += value                       # Same as: total = total + value
print("Total:", total)

### Task 1 — Write `manual_mean()`

Return the arithmetic mean of a list **without** using `sum()` or NumPy. Return `None` for an empty list. Reuse the loop pattern above.

In [ ]:
def manual_mean(values):
    # TODO: Handle an empty list.
    # TODO: Add the values using a for loop.
    # TODO: Return the total divided by the list length.
    raise NotImplementedError("Complete Task 1")

In [ ]:
assert manual_mean([]) is None
assert manual_mean([2, 4, 6]) == 4
assert manual_mean([-2, 0, 2]) == 0
print("Task 1 passed")

### Task 2 — Reuse a function

`count_above_mean(values)` returns how many values are **strictly greater** than their mean. Example: `[3, 7, 2, 9, 4]` has mean `5`; return `2`. Return `0` for an empty list. Call `manual_mean()` instead of calculating the mean again.

In [ ]:
def count_above_mean(values):
    # TODO: Handle the empty case; get the mean; count values above it.
    raise NotImplementedError("Complete Task 2")

In [ ]:
assert count_above_mean([3, 7, 2, 9, 4]) == 2
assert count_above_mean([2, 2, 2]) == 0
assert count_above_mean([-2, -1, -3]) == 1
assert count_above_mean([]) == 0
print("Task 2 passed")

**Next:** Return to the interactive lesson, **Section 02 — NumPy & Shapes**.


## 02 · NumPy & Shapes

NumPy arrays store numerical data and support matrix operations. The interactive lesson shows matrix multiplication first, then broadcasting, then samples and features.

- `(N,)`: a 1D array with `N` values.
- `(N, 1)`: a 2D array with `N` rows and one feature column.
- `(N, 2)`: a 2D array with `N` rows and two feature columns.

`.reshape(-1, 1)` creates a column array; `-1` asks NumPy to work out the row count. `A[:, 0]` selects the first column.


In [ ]:
example = np.array([10., 20., 30.])
example_column = example.reshape(-1, 1)
print("1D:", example.shape)
print("Column:", example_column.shape)
print("First column:", example_column[:, 0])

### Task 3 — Turn animation data into arrays

Create `x` = time in seconds and `y` = horizontal position. Both should be **1D arrays**. Then create `x_column`, a 2D column view of `x`.

**Useful syntax:** `np.array(values, dtype=float)`, division `/ fps`, `.reshape(-1, 1)`.

In [ ]:
# TODO: Make float arrays of frames and positions.
# TODO: Set x to frame numbers divided by fps.
# TODO: Set x_column to a 2D column version of x.
raise NotImplementedError("Complete Task 3")

In [ ]:
N = len(frames)
assert x.shape == (N,) and y.shape == (N,)
assert x_column.shape == (N, 1)
assert np.allclose(x, np.arange(0, 31, 3) / fps)
assert np.allclose(x_column[:, 0], x)
print("Task 3 passed:", "x", x.shape, "x_column", x_column.shape)

### Multiplication: `*` vs `@`

- `A * w`: element-wise multiplication. NumPy may broadcast a vector across rows.
- `A @ w`: matrix multiplication. Each row combines with the weight vector to produce one result.
- `A @ B`: matrix multiplication with two matrices, producing a new matrix.

Check the output shapes as well as the numerical values.


In [ ]:
A_demo = np.array([[1., 2.], [3., 4.], [5., 6.]])  # 3 rows, 2 columns
w_demo = np.array([0.2, 0.8])
print("A * w:\n", A_demo * w_demo)
print("A @ w:", A_demo @ w_demo)
print("Output shape:", (A_demo @ w_demo).shape)

B_demo = np.array([[1., 2.], [0., 1.], [2., 0.]])
C_demo = np.array([[1., 2., 3.], [4., 5., 6.]])
print("Matrix @ matrix:\n", C_demo @ B_demo)
print("Matrix output shape:", (C_demo @ B_demo).shape)


### Task 4 — Predict shapes, then calculate

Write a function that returns **one number per row** of a 2D array. Use `@` rather than a loop or `*`.

In [ ]:
def matrix_product(X, w):
    # TODO: Return matrix multiplication X @ w.
    raise NotImplementedError("Complete Task 4")

In [ ]:
assert matrix_product(A_demo, w_demo).shape == (3,)
assert np.allclose(matrix_product(A_demo, w_demo), [1.8, 3.8, 5.8])
print("Task 4 passed")

**Next:** Return to the interactive lesson, **Section 03 — Prediction & Loss**.


## 03 · Prediction & Loss

With one input feature, the prediction is **ŷ = wx + b**.

- `x`: time in seconds; `w`: weight (slope); `b`: bias (value at `x=0`).
- `y`: measured position; `y_pred` or `ŷ`: predicted position.
- `error = y_pred - y`: the residual for each sample.
- **MSE** (`loss`) = the mean of squared residuals. Smaller MSE means a closer fit **on these samples**.

$$
\mathrm{MSE}=\frac{1}{N}\sum_{i=1}^{N}(\hat{y}_i-y_i)^2
$$

`N` is the number of samples, `i` selects one sample, `ŷᵢ` is its predicted position, `yᵢ` is its measured position, and `Σ` means to sum across all samples.


### Task 5 — Prediction

Complete `predict(x, w, b)`. It should work for one number or a whole NumPy array.

In [ ]:
def predict(x, w, b):
    # TODO: Calculate and return wx + b.
    raise NotImplementedError("Complete Task 5")

In [ ]:
assert np.isclose(predict(2., 3., 1.), 7.)
assert np.allclose(predict(np.array([0., 1.]), 2., 1.), [1., 3.])
print("Task 5 passed")

### Task 6 — MSE: Python first, NumPy next

Given two equal-length arrays, return `mean((y_pred - y)²)`.

**Part A:** use a Python `for` loop and a running total. **Part B:** write the same result with `np.mean()` and array operations. Use the NumPy version later in training.

In [ ]:
def mse_loop(y, y_pred):
    # TODO: Accumulate squared residuals, then divide by the number of samples.
    raise NotImplementedError("Complete Task 6A")

def compute_mse(y, y_pred):
    # TODO: Use np.mean() without a for loop.
    raise NotImplementedError("Complete Task 6B")

In [ ]:
test_y = np.array([1., 2., 4.])
test_pred = np.array([2., 1., 5.])
assert np.isclose(mse_loop(test_y, test_pred), 1.)
assert np.isclose(compute_mse(test_y, test_pred), 1.)
assert np.isclose(compute_mse(y, y), 0.)
print("Task 6 passed")

### See the prediction

The plotting function is already written. Change `w_try` and `b_try` and look at MSE. You do not need to write Matplotlib code.

In [ ]:
def plot_fit(x, y, y_pred, title="Prediction"):
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.scatter(x, y, label="Measured position", s=40, color="#2563eb")
    ax.plot(x, y_pred, label="Predicted position", color="#e68a21", linewidth=2)
    ax.set(xlabel="Time x (seconds)", ylabel="Horizontal position y", title=title)
    ax.grid(alpha=0.25)
    ax.legend()
    plt.show()

w_try, b_try = 0.0, 0.0  # Change these two values
y_try = predict(x, w_try, b_try)
print("MSE:", compute_mse(y, y_try))
plot_fit(x, y, y_try)

**Next:** Return to the interactive lesson, **Section 04 — Gradient Descent**.


## 04 · Gradient Descent

A **derivative** describes how one value changes when another changes. A **partial derivative** describes the change with respect to one parameter while the others stay fixed.

Our `loss` is MSE. For one sample: `y_pred = w*x+b`, `error = y_pred-y`, and squared error `L = error**2`. The interactive lesson explains the chain rule: changing `w` changes `y_pred`, which changes `error`, which changes `L`.

- Single-sample gradients: `∂L/∂w = 2*error*x` and `∂L/∂b = 2*error`.
- MSE averages these contributions over `N` samples, giving the factor **2/N**.

$$
dw=\frac{\partial\mathrm{loss}}{\partial w}=\frac{2}{N}\sum_{i=1}^{N}error_i x_i
$$

$$
db=\frac{\partial\mathrm{loss}}{\partial b}=\frac{2}{N}\sum_{i=1}^{N}error_i
$$

**Why does the main course use 1/(2N)?** Its alternative cost `J = MSE/2` cancels the factor 2 when differentiated. Our code uses `loss = MSE`, so the gradients above keep the 2. Both costs have the same minimum.

**NumPy syntax:** `dw = 2*np.mean(error * x)`; `db = 2*np.mean(error)`.


### Task 7 — Gradients

Complete `compute_gradients()`. Return two **scalar numbers**, `dw` and `db`. Reuse `predict()`.

In [ ]:
def compute_gradients(x, y, w, b):
    y_pred = predict(x, w, b)
    error = y_pred - y
    # TODO: Use the gradient formulas above.
    raise NotImplementedError("Complete Task 7")

In [ ]:
dw_test, db_test = compute_gradients(np.array([0., 1.]), np.array([1., 3.]), 0., 0.)
assert np.isclose(dw_test, -3.) and np.isclose(db_test, -4.)
print("Task 7 passed")

### Task 8 — Update the parameters

Gradient descent moves **against the gradient**:

**new w = old w − learning_rate × dw**  
**new b = old b − learning_rate × db**

The **learning rate** `alpha` controls step size. Calculate both gradients *before* changing either parameter.

In [ ]:
def gradient_step(w, b, dw, db, alpha):
    # TODO: Calculate the new w and b; return both.
    raise NotImplementedError("Complete Task 8")

In [ ]:
w_check, b_check = gradient_step(0., 0., -3., -4., 0.1)
assert np.isclose(w_check, 0.3) and np.isclose(b_check, 0.4)
print("Task 8 passed")

### Task 9 — Training loop

Each **epoch**: **predict → calculate loss → calculate gradients → update parameters**. Store `loss` so we can plot whether training is working.

In [ ]:
def train(x, y, alpha=0.1, epochs=400):
    w, b = 0.0, 0.0
    losses = []
    for epoch in range(epochs):
        # TODO: Predict; record loss; calculate gradients; update w and b.
        raise NotImplementedError("Complete Task 9")
    return w, b, losses

In [ ]:
def plot_losses(losses, title="Loss during training"):
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(losses, color="#0d9488")
    ax.set(xlabel="Epoch", ylabel="MSE", title=title)
    ax.grid(alpha=0.25)
    plt.show()

w_fit, b_fit, losses = train(x, y, alpha=0.1, epochs=400)
assert len(losses) == 400
assert np.isfinite(losses).all()
assert losses[-1] < losses[0] * 0.05
print("Learned w:", round(w_fit, 4), "Learned b:", round(b_fit, 4))
print("MSE before / after:", round(losses[0], 4), round(losses[-1], 4))
plot_fit(x, y, predict(x, w_fit, b_fit), "Trained model")
plot_losses(losses)

### Experiment — Learning rate

The input uses time in seconds (0–1), instead of frame numbers (0–30), so a given learning rate changes the parameters on a more manageable scale.

Change `alpha` to `0.001`, `0.1`, and `1.5` using the next cell. Compare the last MSE. A smaller value may train slowly; a larger value may not converge. (A specific result depends on the data and number of epochs.)

In [ ]:
for alpha_try in [0.001, 0.1, 1.5]:
    w_result, b_result, loss_result = train(x, y, alpha=alpha_try, epochs=400)
    print("alpha:", alpha_try, "last MSE:", round(loss_result[-1], 5))

**Next:** Return to the interactive lesson, **Section 05 — Multiple Features**.


## 05 · Multiple Features

A character moves on a flat **world plane**. There are two input features:

- `world_x`: its horizontal world position.
- `world_y`: its vertical world position in that plane.

Our target, `screen_x`, is one **horizontal screen coordinate** seen by a simplified rotated camera. This is a teaching example of a linear camera mapping, not a full perspective-camera model.

$$
\hat{y}=w_1x_1+w_2x_2+b
$$

In this example: $x_1=$ `world_x`, $x_2=$ `world_y`, and $\hat{y}$ predicts `screen_x`. It is **two inputs and one output**.

NumPy: `y_pred = X @ w + b`, with `X.shape == (N, 2)`, `w.shape == (2,)`, and `y_pred.shape == (N,)`.


In [ ]:
# A second scene: world motion in TWO directions; one screen-position target.
world_x = np.array([0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
world_y = np.array([0.00, 0.28, 0.03, 0.45, 0.18, 0.63, 0.32, 0.74, 0.43, 0.86, 0.62])
screen_x = np.array([1.30, 2.18, 2.00, 3.16, 2.94, 4.16, 3.86, 5.02, 4.72, 5.90, 5.74])
X_world = np.column_stack([world_x, world_y])
print("world_x:", world_x.shape, "world_y:", world_y.shape)
print("X_world:", X_world.shape, "screen_x:", screen_x.shape)
print("First 3 rows of X_world:\n", X_world[:3])
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(world_x, world_y, '-o', label="World movement")
for i, (a, b) in enumerate(zip(world_x, world_y)):
    ax.annotate(str(i), (a, b), textcoords="offset points", xytext=(4, 4), fontsize=8)
ax.set(xlabel="World x", ylabel="World y", title="Movement in two world directions")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

### Task 10 — One output from two features

Implement `predict_multi(X, w, b)` using matrix multiplication and broadcasting. Each row of the input matrix should produce one predicted **screen_x** value.


In [ ]:
def predict_multi(X, w, b):
    # TODO: Return one screen_x prediction per row.
    raise NotImplementedError("Complete Task 10")


In [ ]:
result = predict_multi(X_world, np.array([2., 3.]), 1.0)
assert result.shape == (len(screen_x),)
assert np.allclose(predict_multi(np.array([[1., 2.], [3., 4.]]),
                                 np.array([2., 1.]), 0.5), [4.5, 10.5])
print("Task 10 passed")


### Matrix gradient — prepare for Section 06

Each input feature has its own weight and gradient. The two weight gradients form the vector `dw`.

$$
d\mathbf{w} = \frac{2}{N} X^T\mathbf{error},\qquad db = 2\operatorname{mean}(\mathbf{error})
$$

| Math | NumPy | Shape |
|---|---|---|
| Input matrix | `X_world` | `(N, 2)` |
| Transpose | `X_world.T` | `(2, N)` |
| Error vector | `error` | `(N,)` |
| Both gradients | `(2/N) * (X_world.T @ error)` | `(2,)` |

`X.T` turns feature columns into rows so they can be combined with the error from each sample. The bias gradient remains one number.


In [ ]:
print("X_world shape:", X_world.shape)
print("X_world.T shape:", X_world.T.shape)
error_demo = np.array([1., -1., 2.])
print("First 3 input rows, transposed:", X_world[:3].T.shape)
print("Two feature sums:", X_world[:3].T @ error_demo)


**Next:** Return to the interactive lesson, **Section 06 — Train & Compare**.


## 06 · Train & Compare

Can the model predict `screen_x` more accurately using two world-position features instead of one?

**Step 1:** Train a one-feature model with `world_x` → `screen_x`.

**Step 2:** Train a two-feature model with `X_world = [world_x, world_y]` → `screen_x`.

**Step 3:** Compare both predictions and their MSE curves.

Reuse Section 05 data. The output target stays the same.


In [ ]:
# Reuse the world movement data created in Section 05.
assert X_world.shape == (11, 2) and screen_x.shape == (11,)
print("One-feature input:", world_x.shape)
print("Two-feature input:", X_world.shape)
print("Target screen_x:", screen_x.shape)


### Step 1 — Fit the one-feature model

Reuse `train()` to predict `screen_x` using only `world_x`. Store `w_linear`, `b_linear`, and `linear_losses`.


In [ ]:
# TODO: Use train(world_x, screen_x, ...) and save three results.
raise NotImplementedError("Step 1")


### Step 2 — Fit the two-feature model

Write `train_multi(X, y, ...)`. Initialise a weight vector with `np.zeros(X.shape[1])` and a bias `b = 0.0`.

Each epoch: predict → calculate errors → calculate and store MSE → compute `dw = (2/N) * (X.T @ error)` and `db = 2 * np.mean(error)` → update both weights and the bias. Return `(w, b, losses)`.


In [ ]:
def train_multi(X, y, alpha=0.1, epochs=1500):
    # TODO: Initialise weights (one per feature), bias, and loss list.
    # TODO: Repeat prediction, loss, gradients, and updates every epoch.
    # TODO: Return w, b, losses.
    raise NotImplementedError("Step 2")


### Step 3 — Compare predictions

The plotting helper is provided. Plot the same measured `screen_x` target against both model predictions, with `world_x` on the horizontal axis. Compare the MSE curves.


In [ ]:
def plot_comparison(x_axis, y_true, one_pred, two_pred, one_losses, two_losses):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    axes[0].scatter(x_axis, y_true, label="Measured screen_x", color="#2563eb")
    axes[0].plot(x_axis, one_pred, label="One feature", color="#e68a21")
    axes[0].plot(x_axis, two_pred, label="Two features", color="#0d9488")
    axes[0].set(xlabel="World x coordinate", ylabel="Screen x coordinate",
                title="Screen-position predictions")
    axes[0].legend()
    axes[1].plot(one_losses, label="One feature", color="#e68a21")
    axes[1].plot(two_losses, label="Two features", color="#0d9488")
    axes[1].set(xlabel="Epoch", ylabel="MSE", title="Training loss")
    axes[1].legend()
    for ax in axes: ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

# TODO: Use train_multi(X_world, screen_x, ...) to get w_multi, b_multi, multi_losses.
# TODO: Generate one_pred and two_pred.
# TODO: Call plot_comparison(world_x, screen_x, ...).
raise NotImplementedError("Step 3")


### Check your results

Explain briefly:

- Which model predicts `screen_x` more closely, and why?
- How are **two input features** different from **two output targets**?
- What happens when you change the learning rate?
- Does low MSE on these samples guarantee good predictions in unseen scenes?

**Your observations:**

> Write here.
